## Download data from the ERA5 API

To load ERA5 data from Jupyter NRTBuc using Python, you typically access the data via the CDS (Climate Data Store) API provided by ECMWF. Here's a basic step-by-step guide to do that:

```Step 1: Set Up Your CDS API Key```

Register for a CDS account at ECMWF CDS. url: https://cds.climate.copernicus.eu/

After registration, you can find your url and key here: url: https://cds.climate.copernicus.eu/how-to-api

Set the API key by placing the .cdsapirc file in your home directory (C:\Users\username) with the following content:

```python
url: https://cds.climate.copernicus.eu/api/v2

key: your-api-key
```

```Step 2: Install Required Packages```
You need the cdsapi library to access ECMWF data. You can install it using:

```python
pip install cdsapi
```

### CDS API Request Queue Limit

The Copernicus Climate Data Store (CDS) may temporarily reject data retrieval requests when the maximum number of queued requests for a specific dataset has been reached.

This is a server-side limitation and does not necessarily indicate an error in the request parameters or the downloader itself. Even valid requests may be rejected during periods of high server demand.

**Possible error message:**
```text
Maximum number queued requests for this dataset has been reached.
Please configure your scripts to request data sequentially.
```

**What should you do?**

- Make sure that no multiple instances of the downloader are running simultaneously.
- Avoid submitting multiple requests to the same dataset at the same time.
- If a request is rejected due to the queue limit, wait for some time before trying again.
- Do not repeatedly resubmit rejected requests, as this may increase the load on the CDS service.

The downloader uses sequential requests to reduce the likelihood of reaching this limit. However, queue availability also depends on the overall CDS server workload and requests submitted by other users.

**Note:** This limitation is imposed by the CDS infrastructure and is outside the control of this downloader.

### Recommended Time for Data Downloads

CDS request processing times vary depending on the overall system workload. Although there is no officially guaranteed time when the request queue is least busy, downloads may be processed faster during off-peak hours.

**Recommended practices:**

- Consider submitting large data requests during nighttime or early morning hours (UTC), when user activity may be lower.
- Avoid peak daytime periods when possible.
- Check the current CDS queue status before starting large download operations: [CDS Live](https://cds.climate.copernicus.eu/live).
- Monitor the queue status to identify periods of lower activity.
- For large download operations, consider running the script overnight rather than during working hours.

**Important:** Nighttime downloads are not guaranteed to be faster. CDS workload varies continuously and depends on dataset availability, request size, server capacity, and activity from other users. The recommended time window is a practical suggestion, not an official CDS scheduling policy.

In [8]:
# Helper
year = ["1950", "1951",
        "1952", "1953", "1954",
        "1955", "1956", "1957",
        "1958", "1959", "1960",
        "1961", "1962", "1963",
        "1964", "1965", "1966",
        "1967", "1968", "1969",
        "1970", "1971", "1972",
        "1973", "1974", "1975",
        "1976", "1977", "1978",
        "1979", "1980", "1981",
        "1982", "1983", "1984",
        "1985", "1986", "1987",
        "1988", "1989", "1990",
        "1991", "1992", "1993",
        "1994", "1995", "1996",
        "1997", "1998", "1999",
        "2000", "2001", "2002",
        "2003", "2004", "2005",
        "2006", "2007", "2008",
        "2009", "2010", "2011",
        "2012", "2013", "2014",
        "2015", "2016", "2017",
        "2018", "2019", "2020",
        "2021", "2022", "2023",
        "2024", "2025", "2026"]

month = ["01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"]
days =  ["01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
times = ["00:00", "06:00", "12:00", "18:00"]  # Four time steps per day
area = [90, -180, -90, 180]  # North data
pressure_level_all = [
    1000, 975, 950, 925, 900, 875, 850, 825, 800, 775,
    750, 700, 650, 600, 550, 500, 450, 400, 350, 300,
    250, 225, 200, 175, 150, 125, 100, 70, 50, 30,
    20, 10, 7, 5, 3, 2, 1
] # All available pressure levels in hPa

In [1]:
from  ERA5_loader import *

In [2]:
# Find variables related to temperature
temperature = search_era5_variables("temperature")

There are following variables with keyword 'temperature', level_type:None, parameter_type:None:
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
                                                            name      units                                          cds_variable                 level_type     parameter_type
                                                     Temperature          K                                           temperature            pressure_levels      instantaneous
                                      Lake mix-layer temperature          K                            lake_mix_layer_temperature              single_levels      instantaneous
                                         Lake bottom temperature          K                               lake_bottom_temperature              single_levels      instantaneous
                        

In [3]:
# Search for temperature on pressure levels
temperature_pl = search_era5_variables("temperature", level_type="pressure_levels")

There are following variables with keyword 'temperature', level_type:pressure_levels, parameter_type:None:
-------------------------------------------------------------
       name units cds_variable      level_type parameter_type
Temperature     K  temperature pressure_levels  instantaneous
-------------------------------------------------------------
Note: `cds_variable` is used in CDS API download requests.



In [4]:
temperature_pl = search_era5_variables("rain", level_type="single_levels")

There are following variables with keyword 'rain', level_type:single_levels, parameter_type:None:
-----------------------------------------------------------------------------------------------
                   name          units            cds_variable    level_type     parameter_type
   Convective rain rate kg m**-2 s**-1    convective_rain_rate single_levels      instantaneous
  Large scale rain rate kg m**-2 s**-1   large_scale_rain_rate single_levels      instantaneous
Total column rain water       kg m**-2 total_column_rain_water single_levels vertical_integrals
-----------------------------------------------------------------------------------------------
Note: `cds_variable` is used in CDS API download requests.



In [3]:
out = r"ERA5_test"

In [4]:
request_infos = submit_era5_data_for_period(
    output_dir=out,
    variable="2m_temperature",
    date_from="01-01-2020",
    date_to="03-01-2020",
    logging=True,
    grid_resolution=[2.5, 2.5]
)

2026-11-20 18:00:13,831 - download_era5_data - INFO - The ERA5 reanalysis atmospheric data resolution is 0.25° and the ERA5 ensemble atmospheric data resolution is 0.5°. The ERA5 wave data resolution is 0.5° and the ERA5 ensemble wave data resolution is 1.0°. The ERA5-Land data resolution is on 0.1°. Output grid spacing can be changed using 'grid_resolution'.
 Grid resolution input formats:
  1. List or tuple: [dLon, dLat], e.g. [0.25, 0.25].
  2. Slash-separated string: 'dLon/dLat', e.g. '0.25/0.25'.
  Values must be positive finite numbers in degrees. Note: Values < 0.25° will not add real physical details, instead interpolation will be done on the server.
2026-11-20 18:00:13,831 - download_era5_data - INFO - Selected grid resolution: dLon = 2.5000°, dLat = 2.5000°
2026-11-20 18:00:13,833 - download_era5_data - INFO - Request already exists: be7fd29c-16fb-45c5-ba31-6b9202cb2686 (status: accepted). No new request submitted.
2026-11-20 18:00:13,833 - download_era5_data_for_period - INF

In [2]:
request_db = ERA5RequestDatabase()

pending = request_db.list_requests(status="accepted")
for request in pending:
    request = check_era5_status(request["request_id"], request_db)
    print(
        request["request_id"],
        request["output_file"],
        request['status']
    )

be7fd29c-16fb-45c5-ba31-6b9202cb2686 ERA5_test\2m_temperature_single_hourly_2020-01-01_.grib successful
76fe42e9-8293-4f29-9792-d3f42d2a9285 ERA5_test\2m_temperature_single_hourly_2020-01-02_.grib successful
6718e39b-fb60-4079-be8c-469e760414c9 ERA5_test\2m_temperature_single_hourly_2020-01-03_.grib accepted


In [3]:
download_era5_results(request_db, logging=True)

2026-11-21 00:31:14,667 - download_era5_results - INFO - Request be7fd29c-16fb-45c5-ba31-6b9202cb2686 downloaded: ERA5_test\2m_temperature_single_hourly_2020-01-01_.grib


2026-11-21 00:31:15,568 - download_era5_results - INFO - Request 76fe42e9-8293-4f29-9792-d3f42d2a9285 downloaded: ERA5_test\2m_temperature_single_hourly_2020-01-02_.grib


2026-11-21 00:31:15,841 - download_era5_results - INFO - Request 6718e39b-fb60-4079-be8c-469e760414c9 is not ready. Current status: accepted
2026-11-21 00:31:15,841 - download_era5_results - INFO - Download process completed. Downloaded 2 of 3 requests.


[{'request_id': 'be7fd29c-16fb-45c5-ba31-6b9202cb2686',
  'output_file': 'ERA5_test\\2m_temperature_single_hourly_2020-01-01_.grib',
  'result': 'ERA5_test\\2m_temperature_single_hourly_2020-01-01_.grib',
  'status': 'downloaded'},
 {'request_id': '76fe42e9-8293-4f29-9792-d3f42d2a9285',
  'output_file': 'ERA5_test\\2m_temperature_single_hourly_2020-01-02_.grib',
  'result': 'ERA5_test\\2m_temperature_single_hourly_2020-01-02_.grib',
  'status': 'downloaded'}]